# Noise in quantum machine learning
**Quantum Machine Learning · Week 4 (QML for research) · Day 1**

Lesson page: https://sara-aminpour.com/tutorials/qml/w4d1.html

In [ ]:
# On Google Colab, this installs qiskit and qiskit-aer (about a minute). On your own computer, skip it and
# install the packages once instead: pip install qiskit==2.5.2 qiskit-aer==0.17.2 scikit-learn torch matplotlib
import sys
if "google.colab" in sys.modules:
    %pip install --quiet qiskit==2.5.2 qiskit-aer==0.17.2

**Noise in quantum machine learning.** Real qubits are noisy (QC week 6). What does noise do to a trained quantum
classifier? Some noise only shrinks ⟨Z⟩, some shifts it, and in deep circuits it drowns the gradients.

In [ ]:
import numpy as np
from qiskit import QuantumCircuit
from qiskit.circuit import ParameterVector
from qiskit.quantum_info import SparsePauliOp
from qiskit_aer.primitives import EstimatorV2 as AerEstimator
from qiskit_aer.noise import NoiseModel, depolarizing_error, amplitude_damping_error
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split

def classifier(layers=2, reupload=False):
    x, w = ParameterVector("x", 2), ParameterVector("w", 4 * layers)
    qc = QuantumCircuit(2)
    for l in range(layers):
        if reupload or l == 0:
            qc.ry(x[0], 0)
            qc.ry(x[1], 1)
        qc.ry(w[4 * l], 0)
        qc.rz(w[4 * l + 1], 0)
        qc.ry(w[4 * l + 2], 1)
        qc.rz(w[4 * l + 3], 1)
        qc.cx(0, 1)
    return qc

def estimator(noise_model=None):
    """Exact expectation values; with a noise model, from a density-matrix simulation."""
    options = {"max_parallel_threads": 1}
    if noise_model is not None:
        options.update(method="density_matrix", noise_model=noise_model)
    return AerEstimator(options={"backend_options": options})

def scores(est, circuit, X, weight_sets):
    rows = np.array([np.concatenate([W, x]) for W in weight_sets for x in X])
    return est.run([(circuit, SparsePauliOp("ZI"), rows)]).result()[0].data.evs.reshape(len(weight_sets), len(X))

rng = np.random.default_rng(0)
corner = rng.choice([-1, 1], size=(120, 2))
X, y = corner + 0.35 * rng.normal(size=(120, 2)), (corner[:, 0] * corner[:, 1] < 0).astype(int)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=40, random_state=0, stratify=y)
scaler = MinMaxScaler((-np.pi / 2, np.pi / 2)).fit(X_train)
X_train, X_test = scaler.transform(X_train), scaler.transform(X_test)

def train(est, circuit, X, y, epochs=60, lr=0.3, seed=0):
    """Week 3's training: mean squared error of ⟨Z⟩ against ±1, parameter shift, gradient descent."""
    t = 1 - 2 * y
    W = 0.6 * np.random.default_rng(seed).normal(size=circuit.num_parameters - 2)
    shifts = [s * np.eye(len(W))[j] for j in range(len(W)) for s in (np.pi / 2, -np.pi / 2)]
    for epoch in range(epochs):
        S = scores(est, circuit, X, [W] + [W + d for d in shifts])
        W = W - lr * ((S[1::2] - S[2::2]) / 2) @ (2 * (S[0] - t)) / len(t)
    return W

circuit, clean = classifier(), estimator()
W = train(clean, circuit, X_train, y_train)                        # trained on a perfect simulator
z_clean = scores(clean, circuit, X_test, [W])[0]
print(f"noise-free: test accuracy {np.mean((z_clean < 0) == y_test):.3f}, average |⟨Z⟩| {np.mean(np.abs(z_clean)):.3f}")

## Gate noise: depolarizing after every gate
Each gate is followed by a small chance $p$ of scrambling its qubits (QC week 6).

In [ ]:
def depolarizing(p):
    model = NoiseModel()
    model.add_all_qubit_quantum_error(depolarizing_error(p, 1), ["ry", "rz"])
    model.add_all_qubit_quantum_error(depolarizing_error(p, 2), ["cx"])
    return model

print("   p   test accuracy   average |⟨Z⟩|   predictions changed by the noise")
for p in (0, 0.01, 0.02, 0.05, 0.1):
    z = scores(estimator(depolarizing(p)), circuit, X_test, [W])[0]
    print(f"{p:5.2f}       {np.mean((z < 0) == y_test):.3f}           {np.mean(np.abs(z)):.3f}            {np.sum((z < 0) != (z_clean < 0))}")

## Smaller margins, more shot noise
The noise barely moved the boundary, but it shrank every score toward 0. Now measure with shots
(odd numbers of shots, so there are no ties), 500 times over.

In [ ]:
z_noisy = scores(estimator(depolarizing(0.05)), circuit, X_test, [W])[0]
rng = np.random.default_rng(1)
for shots in (21, 101):
    for name, z in (("noise-free", z_clean), ("p = 0.05  ", z_noisy)):
        ones = rng.binomial(shots, (1 - z) / 2, size=(500, len(z)))
        acc = np.mean((1 - 2 * ones / shots < 0) == y_test, axis=1)
        print(f"{shots:3d} shots, {name}: accuracy {acc.mean():.3f} ± {acc.std():.3f}")

## Relaxation before readout: amplitude damping
A qubit in |1⟩ can decay to |0⟩ while it waits to be measured. That pulls ⟨Z⟩ toward +1:
$\langle Z\rangle \to (1 - \gamma)\langle Z\rangle + \gamma$.

In [ ]:
waiting = circuit.copy()
waiting.id(1)                                                      # the moment before readout, where the decay happens

def damping(gamma):
    model = NoiseModel()
    model.add_quantum_error(amplitude_damping_error(gamma), ["id"], [1])
    return model

print("gamma   test accuracy   average ⟨Z⟩")
for gamma in (0, 0.1, 0.2, 0.3, 0.4):
    z = scores(estimator(damping(gamma)), waiting, X_test, [W])[0]
    print(f"{gamma:5.1f}       {np.mean((z < 0) == y_test):.3f}         {np.mean(z):+.3f}")

## Deep circuits: the gradients fade
The spread of one gradient, $\partial\langle Z\rangle/\partial w_0$, over random weights, as layers are added
(re-uploading in every layer), without and with gate noise.

In [ ]:
import matplotlib.pyplot as plt

def gradient_spread(layers, p, samples=24, seed=0):
    c = classifier(layers, reupload=True)
    est = estimator(depolarizing(p)) if p else clean
    rng = np.random.default_rng(seed)
    point = np.array([[0.4, -0.3]])
    grads = []
    for _ in range(samples):
        Wr = rng.uniform(0, 2 * np.pi, size=4 * layers)
        up, down = Wr.copy(), Wr.copy()
        up[0] += np.pi / 2
        down[0] -= np.pi / 2
        s = scores(est, c, point, [up, down])[:, 0]
        grads.append((s[0] - s[1]) / 2)
    return np.std(grads)

depths = range(1, 11)
fig, ax = plt.subplots(figsize=(6.4, 3.8), layout="constrained")
for p, color in ((0, "#8a85a3"), (0.02, "#7b5cfa"), (0.05, "#ff7a59")):
    spreads = [gradient_spread(L, p) for L in depths]
    ax.semilogy(list(depths), spreads, "o-", color=color, lw=2, label="noise-free" if p == 0 else f"p = {p}")
    print(f"p = {p}: spread of the gradient at 1, 5 and 10 layers:", np.round([spreads[0], spreads[4], spreads[9]], 4))
ax.set_xlabel("layers")
ax.set_ylabel(r"spread of $\partial\langle Z\rangle / \partial w_0$")
ax.legend(frameon=False)
plt.show()

## Your turn: undo the shift
Under amplitude damping with $\gamma = 0.3$, the boundary has moved. Using the *training* points only, find the
threshold $t$ (predict class 1 when ⟨Z⟩ < t) with the best training accuracy, then check the test accuracy.

In [ ]:
damped_train = scores(estimator(damping(0.3)), waiting, X_train, [W])[0]
damped_test = scores(estimator(damping(0.3)), waiting, X_test, [W])[0]
thresholds = np.linspace(-0.5, 0.9, 141)
# TODO: training accuracy for every threshold, pick the best, and compare test accuracy with t = 0 and with it

---
## Solution
Try the exercise yourself first: it's fine to get stuck for a minute. Then compare with this one (there are many good answers).

In [ ]:
damped_train = scores(estimator(damping(0.3)), waiting, X_train, [W])[0]
damped_test = scores(estimator(damping(0.3)), waiting, X_test, [W])[0]
thresholds = np.linspace(-0.5, 0.9, 141)
train_acc = np.array([np.mean((damped_train < t) == y_train) for t in thresholds])
best = thresholds[np.flatnonzero(train_acc == train_acc.max())]
t_star = best.mean()                                               # the middle of the best range
print(f"best thresholds on the training points: {best.min():.2f} to {best.max():.2f}; using t = {t_star:.2f}")
print(f"test accuracy with t = 0   : {np.mean((damped_test < 0) == y_test):.3f}")
print(f"test accuracy with t = {t_star:.2f}: {np.mean((damped_test < t_star) == y_test):.3f}")
print("Damping before readout maps ⟨Z⟩ to 0.7⟨Z⟩ + 0.3, so the boundary moved from 0 to 0.3. A threshold learned on")
print("the noisy device moves it back: this is what the trainable shift b in σ(−a⟨Z⟩ + b) does for free.")